## Define helper functions

In [ ]:
import os
import pickle
from pathlib import Path
import numpy as np

# import required module
import sys

# append the path of the
# parent directory
sys.path.append("..")
 
import env

# Load environment variables
env.load_envs()

# Set the cwd to the project root
PROJECT_ROOT: Path = Path(env.get_env("PROJECT_ROOT"))
assert (
    PROJECT_ROOT.exists()
), "You must configure the PROJECT_ROOT environment variable in a .env file!"

os.chdir(PROJECT_ROOT)

cwd = os.getcwd()

In [27]:
from pymatgen.core.structure import Structure, Lattice
from pymatgen.analysis.structure_matcher import StructureMatcher

# Function to create a structure from unit cell parameters and coordinates
def create_structure(coords, lengths, angles):
    """
    Create a pymatgen Structure object.
    
    Parameters:
        coords (list of list): Fractional coordinates of atoms (e.g., [[0, 0, 0], [0.5, 0.5, 0.5]]).
        lengths (list): Unit cell lengths [a, b, c].
        angles (list): Unit cell angles [alpha, beta, gamma] in degrees.
    
    Returns:
        pymatgen.core.structure.Structure: Generated structure.
    """
    # Create lattice
    lattice = Lattice.from_parameters(*lengths, *angles)
    
    # Create structure (all atoms are Si)
    structure = Structure(lattice, ["Si"] * len(coords), coords)
    return structure

# # Generate structures

def parse_cif(filename):
    with open(f"../../parsed_reconstructions/reconstructions_reconstruct-legacy-150-epochs/{filename}", "r") as f:
        lines = f.readlines()
        lattice = []
        for i in range(6):
            lattice.append(float(lines[i].strip().split(" ")[-1]))

        coords = []
        for line in lines:
            if line.startswith("Si") or line.startswith("Al"):
                coords.append(list(map(float, line.strip().split(" ")[2:5])))

    return lattice, coords

The two structures from recon 1 are not equivalent.
The two structures from recon 2 are considered equivalent.
The two structures from recon 3 are not equivalent.
The two structures from recon 4 are considered equivalent.
The two structures from recon 5 are considered equivalent.
The two structures from recon 6 are considered equivalent.
The two structures from recon 7 are not equivalent.
The two structures from recon 8 are not equivalent.
The two structures from recon 9 are considered equivalent.
The two structures from recon 10 are not equivalent.


## Evaluate reconstructions

In [ ]:
# Check that the first 4 atoms closest to a given one are at a distance around 3 because that suggests they from the tetrahedra that zeolites usually have in their structure
def check_zeolite_validity(structure):
    sorted_distance_matrix = np.sort(structure.distance_matrix, axis=1)

    return np.all([sorted_distance_matrix[i][1:5] < 3.5 for i in range(0, len(sorted_distance_matrix))])

In [ ]:
experiment_name = "reconstruct_zdivae_small"

# artifact_files = retrieve_artifacts_by_name(experiment_name, artifact_type='dataset', project='zeogen', entity='glafk')

# for file in artifact_files:
#     if "samples" in file:
#         with open(file, "rb") as f:
#             samples = pickle.load(f)


reconstructions_file = "reconstructions-reconstruct_zdivae_small.pickle"

with open(f"./reconstructions/{reconstructions_file}", "rb") as f:
    reconstructions = pickle.load(f)
    reconstructions = reconstructions[0]



for i in range(1, len(reconstructions) + 1):
    # # Unit cell data for first structure
    lattice_recon, coords_recon = parse_cif(f"reconstruction_{i}.cif")
    lattice_gt, coords_gt = parse_cif(f"reconstruction_{i}_gt.cif")
    recon = create_structure(coords_recon, lattice_recon[:3], lattice_recon[3:])
    ground_truth = create_structure(coords_gt, lattice_gt[:3], lattice_gt[3:])


    # Initialize StructureMatcher
    matcher = StructureMatcher(ltol=0.3, stol=0.5, angle_tol=10)

    # Compare structures
    is_match = matcher.fit(recon, ground_truth)

    if is_match:
        print(f"The two structures from reconstruction {i} are considered equivalent.")
    else:
        print(f"The two structures from reconstruction {i} are not equivalent.")

    # Check validity
    if check_zeolite_validity(recon):
        print(f"The reconstruction {i} is a valid zeolite.")
    else:
        print(f"The reconstruction {i} is not a valid zeolite.")

    # Optional: Output the similarity transformation
    if is_match:
        transf = matcher.get_mapping(recon, ground_truth)
        print("Transformation matrix:")
        print(transf)

## Evaluate samples

In [ ]:
experiment_name = "sample_zdivae_small"

# artifact_files = retrieve_artifacts_by_name(experiment_name, artifact_type='dataset', project='zeogen', entity='glafk')

# for file in artifact_files:
#     if "samples" in file:
#         with open(file, "rb") as f:
#             samples = pickle.load(f)


samples_file = "samples-sample_zdivae_small.pickle"

with open(f"./samples/{samples_file}", "rb") as f:
    samples = pickle.load(f)
    samples = reconstructions[0]

for i in range(1, len(samples) + 1):
    lengths = samples[i]["lengths"]
    angles = samples[i]["angles"]
    coords = samples[i]["frac_coords"]

    structure = create_structure(coords, lengths, angles)

    if check_zeolite_validity(structure):
        print(f"The sample {i} is a valid zeolite.")
    else:
        print(f"The sample {i} is not a valid zeolite.") 